# Environment Setup & Pure Pathlib Imports

In [4]:
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass


In [5]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

# SRE FIX: Pure pathlib root directory resolution
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from openai import OpenAI
from prefect.blocks.system import Secret

from src.configs import load_settings, root_path
from src.elt.ingestion import execute_ingestion, trigger_nextcloud_occ_scan
from src.elt.scanner import calculate_blake3, scan_sources
from src.elt.strategy import generate_strategy
from src.pipeline.knowledge import run_knowledge_pipeline
from src.pipeline.recovery import run_snapshot_audit
from src.state.schema import (
    get_unified_staging_view,
    init_schema,
    reconcile_deletions,
)

settings = load_settings()

# Pathlib test configurations
test_db = root_path("data/test_ledger.db")
mock_gdrive = root_path("data/mock/mock_gdrive")
mock_onedrive = root_path("data/mock/mock_onedrive")
nextcloud_mount = Path("/nextcloud_data") 

print("✅ Master Life-Cycle Test Suite Initialised.")

✅ Master Life-Cycle Test Suite Initialised.


# 0. Initialize Schema & Vault Auth

In [7]:
print("="*60)
print("🌟 STEP 0: NEXTCLOUD INITIALIZATION")
print("="*60)

# Clean up stale test database using pathlib.Path.unlink()
test_db.unlink(missing_ok=True)
print(f"🧹 Cleaned up stale database: {test_db}")

cloud_sources = {
    "Google Drive": str(mock_gdrive),
    "OneDrive": str(mock_onedrive)
}
init_schema(test_db, list(cloud_sources.keys()))

print("\n🔐 Authenticating with Prefect Vault...")
llm_key = await Secret.load(settings["llm"]["secret_blocks"]["gemini_api_key"])
llm_client = OpenAI(api_key=llm_key.get(), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

print("✅ Step 0: Schema and Vault Authentication Complete.")

🌟 STEP 0: NEXTCLOUD INITIALIZATION
🧹 Cleaned up stale database: /home/coder/projects/agentic-nas-workflow/data/test_ledger.db
Schema initialized. Staging tables created for: ['Google Drive', 'OneDrive']

🔐 Authenticating with Prefect Vault...
✅ Step 0: Schema and Vault Authentication Complete.


# 1. Seed Initial Bulk Cloud Files

In [10]:
print("="*60)
print("🌱 STEP 1: SEEDING INITIAL CLOUD SOURCES")
print("="*60)

for d in [mock_gdrive, mock_onedrive]:
    d.mkdir(parents=True, exist_ok=True)

# Generate mock files using pure Path.write_text()
(mock_gdrive / "2026_tax_return.pdf").write_text("CONFIDENTIAL TAX RETURN 2026")
(mock_gdrive / "aws_july_invoice.txt").write_text("AWS CLOUD INVOICE: $150.00")
(mock_gdrive / "project_alpha_spec.docx").write_text("PROJECT ALPHA SPECIFICATION")

# EXACT CROSS-DRIVE DUPLICATE:
(mock_onedrive / "2026_tax_return.pdf").write_text("CONFIDENTIAL TAX RETURN 2026") 
(mock_onedrive / "family_vacation.jpg").write_text("[IMAGE DATA: SYDNEY BEACH]")

print("📁 Google Drive seeded with 3 files.")
print("📁 OneDrive seeded with 2 files (1 exact cross-drive duplicate).")

🌱 STEP 1: SEEDING INITIAL CLOUD SOURCES
📁 Google Drive seeded with 3 files.
📁 OneDrive seeded with 2 files (1 exact cross-drive duplicate).


# 2. Execute Bulk Scan, Strategy, and Ingest

In [11]:
print("=" * 60)
print("🚀 DAY 0: EXECUTING BULK ELT PIPELINE")
print("=" * 60)

# Phase 1: Multi-Source Scanning (BLAKE3 Hashing)
scan_sources(test_db, cloud_sources)

# Phase 2: Cross-Table Dedupe & LLM Routing Strategy
routings = generate_strategy(
    db_path=test_db, 
    sources=list(cloud_sources.keys()), 
    llm_client=llm_client, 
    model_name=settings["llm"]["model"]
)

# Phase 3: Physical Ingestion to Nextcloud ZFS
execute_ingestion(
    db_path=test_db, 
    nextcloud_mount=nextcloud_mount, 
    routings=routings
)

# Verify Staging & Production DB State
conn = sqlite3.connect(test_db)
prod_count = conn.execute("SELECT COUNT(*) FROM production_inventory").fetchone()[0]
gdrive_dupes = conn.execute("SELECT COUNT(*) FROM staging_google_drive WHERE status='duplicate'").fetchone()[0]
onedrive_dupes = conn.execute("SELECT COUNT(*) FROM staging_onedrive WHERE status='duplicate'").fetchone()[0]
conn.close()

print("\n📊 Day 0 Bulk Ingest Summary:")
print(f"   - Total Unique Files Ingested to Production: {prod_count} (Expected: 4)")
print(f"   - Cross-Drive Duplicates Skipped: {gdrive_dupes + onedrive_dupes} (Expected: 1)")

🚀 DAY 0: EXECUTING BULK ELT PIPELINE
🔍 Scanning source: Google Drive at /home/coder/projects/agentic-nas-workflow/data/mock/mock_gdrive...
   ✅ Google Drive: 8 files staged.
🔍 Scanning source: OneDrive at /home/coder/projects/agentic-nas-workflow/data/mock/mock_onedrive...
   ✅ OneDrive: 4 files staged.
📊 Taxonomy Cache Results | Hits: 0 | Misses (Sent to LLM): 9
🧠 Asking Agent to route 9 unseen files...
🚀 Ingesting: aws_invoice_july.txt -> /Documents/Financial/Invoices/aws_invoice_july.txt
🚀 Ingesting: project_alpha_spec.docx -> /Documents/Work_Projects/project_alpha_spec.docx
🚀 Ingesting: family_photo.jpg -> /Media/Photos/Family/family_photo.jpg
🚀 Ingesting: photo.jpg -> /Media/Photos/Family/photo.jpg
🚀 Ingesting: 2026_tax_return.pdf -> /Documents/Financial/Taxes/2026_tax_return.pdf
🚀 Ingesting: tax_2026.pdf -> /Documents/Financial/Taxes/tax_2026.pdf
   [Dedupe] 🛑 File or Hash already exists in production. Skipping copy.
🚀 Ingesting: report.pdf -> /Documents/Work_Projects/report.pdf
